# Does the hold-out track record agree with the scorecard?


**The check.** Instruction 62 built a per-gene hold-out record for every label strategy and required that, pooled to the strategy level, it agree with the scorecard the strategy cards already show. That agreement was never measured; this measures it.

**Like with like.** The scorecard metric of most label strategies is *correct calls per hidden gene* (an abstention counts as not correct); three are scored by *precision of calls* (abstentions left out). The track record is reduced to the same quantity for each, at the same setting, per organism and target. The two use different folds and seeds, so they should agree within sampling error, not to the digit.

In [1]:
import json, os
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from parascape import track_record as T
runs = pd.read_json(os.path.join(REPO, 'results/calibration_2026-09-26b/runs.jsonl'), lines=True)
ledger = pd.concat([T.shipped(o) for o in ('Tg', 'Pf')], ignore_index=True)
ledger = ledger[ledger['mode'].astype(str) == 'together']
for c in ('organism', 'strategy', 'target', 'setting_key'):
    ledger[c] = ledger[c].astype(str)
print(len(runs), 'calibration runs;', len(ledger), 'hold-out rows over',
      ledger.strategy.nunique(), 'strategies and', ledger.target.nunique(), 'targets')

7640 calibration runs; 391370 hold-out rows over 14 strategies and 12 targets


## 1 · The same quantity from both

A calibration run records only the parameters it varied, and writes floats unrounded (`min_share=0.30000000000000004`); the record writes the full setting. A calibration run is matched to a record setting when every parameter it names has the same value there, after rounding. A record setting the calibration never tried has no scorecard cell, and is counted as such rather than compared with a neighbour.

In [2]:
def norm(v):
    try:
        return str(round(float(v), 6)).rstrip('0').rstrip('.') if not isinstance(v, bool) else str(v)
    except (TypeError, ValueError):
        return str(v)
def parse(key):
    return {k.strip(): norm(v) for k, v in (part.split('=', 1) for part in key.split(', ') if '=' in part)}
lab = runs[runs.strategy.isin(set(ledger.strategy)) & runs.verdict.isin(['PASS', 'FAIL'])].copy()
lab['target'] = lab.settings.map(lambda d: str(d.get('target', '')))
lab['named'] = lab.settings.map(lambda d: {k: norm(v) for k, v in d.items() if k != 'target'})
record_keys = ledger[['organism', 'strategy', 'setting_key']].drop_duplicates()
def match(row):
    keys = record_keys[(record_keys.organism == row.organism) & (record_keys.strategy == row.strategy)].setting_key
    hits = [k for k in keys if all(parse(k).get(p) == v for p, v in row.named.items())]
    return hits[0] if len(hits) == 1 else ''
lab['setting_key'] = lab.apply(match, axis=1)
unmatched = lab[lab.setting_key == ''].groupby(['organism', 'strategy']).size().rename('runs at settings the record does not hold')
lab = lab[lab.setting_key != '']
PRECISION = set(lab.loc[lab.metric.str.startswith('precision'), 'strategy'])
card = (lab.groupby(['organism', 'strategy', 'target', 'setting_key', 'metric'])
        ['observed'].agg(['mean', 'std', 'size']).rename(columns={'mean': 'scorecard', 'std': 'scorecard_sd', 'size': 'runs'})
        .reset_index())
never = sorted(set(map(tuple, record_keys[['organism', 'strategy']].values)) - set(map(tuple, card[['organism', 'strategy']].values)))
print('record strategies whose recorded setting was never calibrated:', never)
def record(part):
    n = len(part); answered = int((~part.abstained.astype(bool)).sum())
    right = int(part.correct.fillna(False).astype(bool).sum())
    return pd.Series(dict(genes=n, answered=answered, right=right))
rec = ledger.groupby(['organism', 'strategy', 'target', 'setting_key']).apply(record).reset_index()
both = card.merge(rec, on=['organism', 'strategy', 'target', 'setting_key'], how='inner')
both['track_record'] = np.where(both.strategy.isin(PRECISION), both.right / both.answered.replace(0, np.nan),
                                both.right / both.genes)
both['difference'] = both.track_record - both.scorecard
# Sampling error of the record itself (binomial) and of the scorecard (spread over its runs).
n = np.where(both.strategy.isin(PRECISION), both.answered, both.genes)
se = np.sqrt(both.track_record * (1 - both.track_record) / np.clip(n, 1, None) + (both.scorecard_sd.fillna(0) ** 2) / both.runs)
both['z'] = both.difference / np.where(se > 0, se, np.nan)
print(len(both), 'organism x strategy x target x setting cells joined;',
      int(card.shape[0] - both.shape[0]), 'scorecard cells have no matching record (other settings)')
both.round(3).head(12)

record strategies whose recorded setting was never calibrated: [('Pf', 'cluster_guilt'), ('Pf', 'physical_partners'), ('Pf', 'supervised_classifier'), ('Tg', 'cluster_guilt'), ('Tg', 'physical_partners'), ('Tg', 'supervised_classifier')]
67 organism x strategy x target x setting cells joined; 20 scorecard cells have no matching record (other settings)


,organism,strategy,target,setting_key,metric,scorecard,scorecard_sd,runs,genes,answered,right,track_record,difference,z
0,Pf,feature_knn,is_exported,"k=15, min_share=0.3",correct calls per hidden gene,0.93,0.041,5,5720,5720,5431,0.949,0.02,1.06
1,Pf,feature_knn,lopit_pf_location,"k=15, min_share=0.3",correct calls per hidden gene,0.489,0.021,5,1580,1264,777,0.492,0.003,0.169
2,Pf,feature_knn,pb_transferred_phenotype,"k=15, min_share=0.3",correct calls per hidden gene,0.648,0.008,5,2433,2433,1582,0.65,0.002,0.212
3,Pf,graph_convolution,is_exported,"C=0.5, hops=2, min_probability=0.0",correct calls per hidden gene,0.916,0.038,5,5720,5720,5320,0.93,0.014,0.833
4,Pf,graph_convolution,lopit_pf_location,"C=0.5, hops=2, min_probability=0.0",correct calls per hidden gene,0.602,0.02,5,1580,1580,938,0.594,-0.008,-0.518
5,Pf,graph_convolution,pb_transferred_phenotype,"C=0.5, hops=2, min_probability=0.0",correct calls per hidden gene,0.575,0.018,5,2433,2433,1401,0.576,0.001,0.064
6,Pf,layer_propagation,is_exported,"layer=coexpression, restart=0.5",correct calls per hidden gene,0.733,0.052,5,5720,5514,4195,0.733,0,0.01
7,Pf,layer_propagation,lopit_pf_location,"layer=coexpression, restart=0.5",correct calls per hidden gene,0.173,0.009,5,1580,1498,268,0.17,-0.003,-0.298
8,Pf,layer_propagation,pb_transferred_phenotype,"layer=coexpression, restart=0.5",correct calls per hidden gene,0.433,0.018,5,2433,2330,1059,0.435,0.003,0.209
9,Pf,layer_vote,is_exported,k=15,correct calls per hidden gene,0.845,0.174,5,5720,4773,4502,0.787,-0.058,-0.738


## 2 · Do they agree?

Agreement is read three ways: the rank correlation across cells (does the record order the strategies the way the scorecard does), the typical absolute difference, and how many cells differ by more than sampling error explains (|z| > 3).

In [3]:
rho = spearmanr(both.track_record, both.scorecard, nan_policy='omit').statistic
mad = float(both.difference.abs().median())
outliers = both[both.z.abs() > 3].sort_values('z', key=abs, ascending=False)
check = pd.DataFrame([dict(cells=len(both), spearman=round(float(rho), 3),
                           median_abs_difference=round(mad, 3),
                           mean_difference=round(float(both.difference.mean()), 3),
                           beyond_sampling_error=len(outliers),
                           agrees=bool(rho >= 0.9 and mad <= 0.05))])
both.to_csv(os.path.join(OUT, 'cells.tsv'), sep='\t', index=False)
check.to_csv(os.path.join(OUT, 'check.tsv'), sep='\t', index=False)
check

,cells,spearman,median_abs_difference,mean_difference,beyond_sampling_error,agrees
0,67,0.875,0.009,-0.015,8,False


In [4]:
outliers[['organism', 'strategy', 'target', 'setting_key', 'scorecard', 'track_record', 'difference', 'z', 'runs', 'genes']].round(3).head(20)

,organism,strategy,target,setting_key,scorecard,track_record,difference,z,runs,genes
45,Tg,map_neighbours,dtm_class,"k=15, min_dist=0.1, min_share=0.3, n_neighbors=25, sample=4000",0.745,0.232,-0.513,-101,5,8137
12,Pf,map_neighbours,is_exported,"k=15, min_dist=0.1, min_share=0.3, n_neighbors=25, sample=4000",0.934,0.423,-0.511,-30.75,5,5720
21,Pf,stratum_focus,is_exported,"k=15, min_share=0.3, stratum=lineage-specific",1,0.949,-0.051,-17.45,1,5720
46,Tg,map_neighbours,lopit_unified,"k=15, min_dist=0.1, min_share=0.3, n_neighbors=25, sample=4000",0.412,0.217,-0.195,-15.38,5,3818
14,Pf,map_neighbours,pb_transferred_phenotype,"k=15, min_dist=0.1, min_share=0.3, n_neighbors=25, sample=4000",0.585,0.462,-0.123,-11.38,5,2433
44,Tg,map_neighbours,compartment,"k=15, min_dist=0.1, min_share=0.3, n_neighbors=25, sample=4000",0.252,0.131,-0.121,-9.287,5,3804
57,Tg,stratum_focus,dtm_class,"k=15, min_share=0.3, stratum=lineage-specific",0.736,0.766,0.03,4.517,5,8137
58,Tg,stratum_focus,lopit_unified,"k=15, min_share=0.3, stratum=lineage-specific",0.468,0.508,0.04,4.081,5,3818


### The one systematic difference: map neighbours counts only the genes it placed

The recorded map-neighbours setting draws its map from a sample of 4,000 genes, so a hidden gene outside the sample is never placed and gets no answer. The record counts it as a hidden gene without a correct call; the scorecard counted only the genes on the map. Over the genes it placed, the record agrees with the scorecard. Both are true; the scorecard line for map neighbours should say *over the genes on the sampled map*, because it overstates how many hidden genes the strategy answers.

In [5]:
mn = both[both.strategy == 'map_neighbours'].copy()
mn['placed_share'] = mn.answered / mn.genes
mn['record_over_placed'] = mn.right / mn.answered
aligned = both.copy()
sel = aligned.strategy == 'map_neighbours'
aligned.loc[sel, 'track_record'] = aligned.loc[sel, 'right'] / aligned.loc[sel, 'answered']
aligned['difference'] = aligned.track_record - aligned.scorecard
rho2 = spearmanr(aligned.track_record, aligned.scorecard).statistic
mad2 = float(aligned.difference.abs().median())
check2 = pd.DataFrame([dict(cells=len(aligned), spearman=round(float(rho2), 3), median_abs_difference=round(mad2, 3),
                            mean_difference=round(float(aligned.difference.mean()), 3),
                            within_005=int((aligned.difference.abs() <= 0.05).sum()),
                            agrees=bool(rho2 >= 0.9 and mad2 <= 0.05))])
check_all = pd.concat([check.assign(reading='as recorded'), check2.assign(reading='map neighbours over the genes it placed')], ignore_index=True)
check_all.to_csv(os.path.join(OUT, 'check.tsv'), sep='\t', index=False)
display_cols = ['organism', 'target', 'scorecard', 'track_record', 'placed_share', 'record_over_placed']
print(mn[display_cols].round(3).to_string(index=False))
check_all

organism                   target  scorecard  track_record  placed_share  record_over_placed
      Pf              is_exported      0.934         0.423         0.437               0.967
      Pf        lopit_pf_location      0.341         0.348         0.765               0.455
      Pf pb_transferred_phenotype      0.585         0.462         0.772               0.599
      Tg              compartment      0.252         0.131         0.361               0.364
      Tg                dtm_class      0.745         0.232         0.307               0.756
      Tg            lopit_unified      0.412         0.217         0.471               0.459
      Tg     screen_any_phenotype      0.638         0.645         1.000               0.645


,cells,spearman,median_abs_difference,mean_difference,beyond_sampling_error,agrees,reading,within_005
0,67,0.875,0.009,-0.015,8,False,as recorded,NaN
1,67,0.988,0.011,0.012,NaN,True,map neighbours over the genes it placed,62


### By strategy

Pooled to the strategy level -- the level the requirement names -- with the cells it rests on.

In [6]:
by_strategy = both.groupby(['organism', 'strategy']).agg(cells=('target', 'size'),
    scorecard=('scorecard', 'mean'), track_record=('track_record', 'mean')).reset_index()
by_strategy['difference'] = by_strategy.track_record - by_strategy.scorecard
by_strategy.to_csv(os.path.join(OUT, 'by_strategy.tsv'), sep='\t', index=False)
by_strategy.round(3)

,organism,strategy,cells,scorecard,track_record,difference
0,Pf,feature_knn,3,0.689,0.697,0.008
1,Pf,graph_convolution,3,0.697,0.7,0.002
2,Pf,layer_propagation,3,0.446,0.446,-0
3,Pf,layer_vote,3,0.675,0.662,-0.013
4,Pf,map_neighbours,3,0.62,0.411,-0.209
5,Pf,random_forest,3,0.749,0.757,0.008
6,Pf,stacking,3,0.698,0.707,0.01
7,Pf,stratum_focus,1,1,0.949,-0.051
8,Pf,triangulation,3,0.757,0.772,0.015
9,Pf,understudied_first,3,0.761,0.772,0.011
